# Portfolio Optimization Project: V1 Implementation

This notebook implements Version 1 (V1) of a binary portfolio optimization system, focusing on an objective function and a cardinality constraint. It is designed to be beginner-friendly and mathematically verifiable.

## Section 1: Imports

In [182]:
# Import necessary libraries
import numpy as np
import pandas as pd
import itertools
import matplotlib.pyplot as plt # Optional, for basic plotting if desired

## Section 2: Load uploaded files

In [183]:
import pandas as pd
from google.colab import files
import io

print("Please upload 'mu.csv'.")
# Upload mu.csv
uploaded_mu = files.upload()
# Get the actual filename Colab used for mu.csv (e.g., 'mu.csv' or 'mu (1).csv')
mu_actual_filename = list(uploaded_mu.keys())[0]
mu_file_content = uploaded_mu[mu_actual_filename].decode('utf-8')
mu = pd.read_csv(io.StringIO(mu_file_content), index_col=0).squeeze() # Squeeze to get a Series/1D array

print("Please upload 'Sigma.csv'.")
# Upload Sigma.csv
uploaded_sigma = files.upload()
# Get the actual filename Colab used for Sigma.csv
sigma_actual_filename = list(uploaded_sigma.keys())[0]
sigma_file_content = uploaded_sigma[sigma_actual_filename].decode('utf-8')
Sigma = pd.read_csv(io.StringIO(sigma_file_content), index_col=0)

print("Files uploaded and loaded successfully.")

Please upload 'mu.csv'.


Saving mu.csv to mu (6).csv
Please upload 'Sigma.csv'.


Saving Sigma.csv to Sigma (6).csv
Files uploaded and loaded successfully.


In [184]:
# Perform sanity checks

# Print mu shape
print(f"mu shape: {mu.shape}")

# Print Sigma shape
print(f"Sigma shape: {Sigma.shape}")

# Check for missing values
if mu.isnull().any() or Sigma.isnull().any().any():
    print("Warning: Missing values detected in mu or Sigma.")
else:
    print("No missing values detected.")

# Expected shapes
expected_mu_shape = (20,)
expected_sigma_shape = (20, 20)

if mu.shape == expected_mu_shape and Sigma.shape == expected_sigma_shape:
    print(f"Shapes match expected: mu {expected_mu_shape}, Sigma {expected_sigma_shape}.")
else:
    print(f"Warning: Shapes do not match expected. Expected mu {expected_mu_shape}, Sigma {expected_sigma_shape}. Actual mu {mu.shape}, Sigma {Sigma.shape}.")

mu shape: (20,)
Sigma shape: (20, 20)
No missing values detected.
Shapes match expected: mu (20,), Sigma (20, 20).


## Section 2.5: Stock and Sector Definitions

In [185]:
stocks = [
    "AAPL","MSFT","NVDA","GOOGL","AMZN", # Technology
    "JPM","V","MA","GS","BAC", # Finance
    "LLY","JNJ","MRK","PFE","ABBV", # Healthcare
    "XOM","CVX","GE","CAT","HON" # Industrial_Energy
]

# Define the sector map based on the 20 stocks
sector_map = {
    "Technology": list(range(0, 5)), # Indices 0-4
    "Finance": list(range(5, 10)),  # Indices 5-9
    "Healthcare": list(range(10, 15)), # Indices 10-14
    "Industrial_Energy": list(range(15, 20)) # Indices 15-19
}

print("Stock list defined.")
print("Sector map defined:", sector_map)

Stock list defined.
Sector map defined: {'Technology': [0, 1, 2, 3, 4], 'Finance': [5, 6, 7, 8, 9], 'Healthcare': [10, 11, 12, 13, 14], 'Industrial_Energy': [15, 16, 17, 18, 19]}


## Section 3: Define frozen parameters

In [186]:
# Define frozen parameters for the optimization problem

# K: Target portfolio size (number of stocks to select)
K = 6
print(f"Target portfolio size (K): {K}")

# lambda_values: Risk-aversion parameters to sweep through
# Lower lambda means more emphasis on expected return, higher lambda emphasizes diversification.
lambda_values = [0.1, 0.5, 1.0, 2.0]
print(f"Lambda values for sweep: {lambda_values}")

# alpha: Cardinality penalty coefficient
# This parameter penalizes portfolios that do not meet the target cardinality K.
alpha = 20
print(f"Cardinality penalty coefficient (alpha): {alpha}")

# Number of stocks (derived from mu length, assuming it's consistent with Sigma)
n = len(mu)
print(f"Total number of available stocks (n): {n}")

Target portfolio size (K): 6
Lambda values for sweep: [0.1, 0.5, 1.0, 2.0]
Cardinality penalty coefficient (alpha): 20
Total number of available stocks (n): 20


In [187]:
# C: Maximum number of stocks allowed per sector
# This is the 'sector_cap' parameter.
sector_cap = 2
print(f"Maximum stocks per sector (sector_cap, C): {sector_cap}")

# beta_values: Sector penalty coefficients to sweep through
# Higher beta means more emphasis on adhering to sector cap.
beta_values = [0, 0.1, 1, 10]
print(f"Beta values for sweep: {beta_values}")

Maximum stocks per sector (sector_cap, C): 2
Beta values for sweep: [0, 0.1, 1, 10]


In [188]:
# ==========================================
# V3 — Budget Parameters
# ==========================================

print("\n--- V3 Budget Parameters ---")

# Frozen stock prices
prices = np.array([
    210, 420, 1200, 180, 220,
    250, 320, 520, 610, 45,
    780, 155, 110, 30, 205,
    115, 145, 200, 390, 470
])

print("Stock prices loaded.")

# Fixed budget
budget = 1100

print(f"Budget: ${budget}")

# Budget penalty sweep
gamma_values = [0, 0.01, 0.1, 1]

print(f"Gamma values: {gamma_values}")


--- V3 Budget Parameters ---
Stock prices loaded.
Budget: $1100
Gamma values: [0, 0.01, 0.1, 1]


## Section 4: Define energy/objective function

In [189]:
def energy(
    x,
    mu,
    Sigma,
    lam,
    alpha,
    K,
    sector_map=None,
    sector_cap=2,
    beta=0,
    prices=None,  # New parameter
    budget=None,  # New parameter
    gamma=0       # New parameter
):
    mu = np.array(mu)
    Sigma = np.array(Sigma)
    x = np.array(x)

    # Return term
    term1 = -np.dot(mu, x)

    # Risk term
    term2 = lam * np.dot(x.T, np.dot(Sigma, x))

    # Cardinality term
    term3 = alpha * (np.sum(x) - K)**2

    # Sector penalty
    term4 = 0
    if sector_map is not None and beta > 0:
        term4 = sector_penalty(
            x,
            sector_map,
            sector_cap,
            beta
        )

    # Budget penalty
    term5 = 0
    if prices is not None and budget is not None and gamma > 0:
        term5 = budget_penalty(
            x,
            prices,
            budget,
            gamma
        )

    total_energy = term1 + term2 + term3 + term4 + term5

    return total_energy

In [190]:
# Helper function to get sector counts for a given portfolio
def get_sector_counts(portfolio_x, sector_map):
    counts = {}
    for sector_name, indices in sector_map.items():
        counts[sector_name] = np.sum(portfolio_x[indices])
    return counts

# Helper function to check for sector cap violation
def check_sector_violation(sector_counts, sector_cap):
    violations = {sector: count for sector, count in sector_counts.items() if count > sector_cap}
    if violations:
        return f"Violated: {violations}"
    else:
        return "None"

In [191]:
def sector_penalty(x, sector_map, sector_cap, beta):
    """
    Computes the sector exposure penalty for a given portfolio x.

    P_sec(x) = beta * sum_j max(0, sector_count_j - sector_cap)^2

    Args:
        x (np.array): A binary vector representing the portfolio.
        sector_map (dict): A dictionary mapping sector names to lists of stock indices.
        sector_cap (int): The maximum number of stocks allowed per sector.
        beta (float): The sector penalty coefficient.

    Returns:
        float: The computed sector penalty.
    """
    penalty = 0.0
    for sector_name, indices in sector_map.items():
        # Count selected stocks in the current sector
        sector_count_j = np.sum(x[indices])
        # Calculate violation
        violation = max(0, sector_count_j - sector_cap)
        # Add to total penalty
        penalty += beta * (violation**2)
    return penalty

In [192]:
def budget_penalty(
    x,
    prices,
    budget,
    gamma
):
    """
    Computes the soft budget penalty.

    P_budget(x)
    =
    gamma * max(
        0,
        total_cost - budget
    )^2

    Args:
        x (np.array):
            Binary portfolio vector.

        prices (np.array):
            Stock prices.

        budget (float):
            Maximum allowed budget.

        gamma (float):
            Budget penalty coefficient.

    Returns:
        float:
            Budget penalty value.
    """

    # Total cost of selected portfolio
    total_cost = np.dot(prices, x)

    # Budget violation amount
    violation = max(
        0,
        total_cost - budget
    )

    # Quadratic penalty
    penalty = gamma * (violation ** 2)

    return penalty

## Section 5: Tiny verification stage

In [193]:
# This tiny verification stage is crucial for ensuring the objective function and basic logic
# are correctly implemented before running on the full dataset. It allows us to manually
# check results for a small, exhaustive search space.

# Take only the first 5 stocks for this small test
num_stocks_small = 5
mu_small = mu[:num_stocks_small]
Sigma_small = Sigma.iloc[:num_stocks_small, :num_stocks_small]

# Set a smaller target portfolio size for the test
K_small = 2

# Use an arbitrary lambda for verification
lam_small = 0.5

print(f"--- Tiny Verification Stage (first {num_stocks_small} stocks, K={K_small}, lambda={lam_small}) ---")
print(f"mu_small shape: {mu_small.shape}")
print(f"Sigma_small shape: {Sigma_small.shape}")

# Generate all binary portfolios for these 5 stocks (2^5 = 32 possibilities)
# Using itertools.product for a brute-force check on a small scale.
all_portfolios_small = list(itertools.product([0, 1], repeat=num_stocks_small))

min_energy_small = float('inf')
best_portfolio_small = None

# Evaluate all 32 possibilities
for portfolio_x in all_portfolios_small:
    current_energy = energy(portfolio_x, mu_small, Sigma_small, lam_small, alpha, K_small)
    if current_energy < min_energy_small:
        min_energy_small = current_energy
        best_portfolio_small = portfolio_x

stocks = [
    "AAPL","MSFT","NVDA","GOOGL","AMZN",
    "JPM","V","MA","GS","BAC",
    "LLY","JNJ","MRK","PFE","ABBV",
    "XOM","CVX","GE","CAT","HON"
]

selected_indices_small = [i for i, x_val in enumerate(best_portfolio_small) if x_val == 1]
selected_stocks_names_small = [
    stocks[i] for i in selected_indices_small
] # Assuming generic stock names
cardinality_small = np.sum(best_portfolio_small)

print("\nVerification Results:")
print(f"Best portfolio (binary representation): {best_portfolio_small}")
print(f"Selected stocks (indices): {selected_indices_small}")
print(f"Selected stocks (names): {selected_stocks_names_small}")
print(f"Minimum energy: {min_energy_small:.4f}")
print(f"Cardinality: {cardinality_small}")
print(f"Expected cardinality (K_small): {K_small}")

if cardinality_small == K_small:
    print("Cardinality constraint is satisfied for the best portfolio in verification (good).")
else:
    print("Cardinality constraint is NOT satisfied for the best portfolio in verification (check alpha).")

--- Tiny Verification Stage (first 5 stocks, K=2, lambda=0.5) ---
mu_small shape: (5,)
Sigma_small shape: (5, 5)

Verification Results:
Best portfolio (binary representation): (1, 0, 0, 0, 1)
Selected stocks (indices): [0, 4]
Selected stocks (names): ['AAPL', 'AMZN']
Minimum energy: -0.0055
Cardinality: 2
Expected cardinality (K_small): 2
Cardinality constraint is satisfied for the best portfolio in verification (good).


## Section 6: Full V1 implementation

In [194]:
# For the full V1 implementation, we will NOT brute force all 2^n portfolios.
# Instead, we generate ONLY feasible portfolios satisfying `sum(x) == K`.
# This is done using `itertools.combinations`, which is computationally much smarter
# for sparse binary vectors with a fixed number of ones.

print(f"--- Full V1 Implementation (n={n} stocks, target K={K}) ---")

# Generate all combinations of K stock indices out of n available stocks.
# Each combination represents a feasible portfolio that satisfies sum(x) == K.

# This function will be called repeatedly in the lambda sweep, so we'll define a helper here.
def find_best_portfolio_for_lambda(
    current_lambda,
    current_beta,
    mu,
    Sigma,
    alpha,
    K,
    n,
    sector_map,
    sector_cap,
    stock_names,
    prices,        # New parameter
    budget,        # New parameter
    current_gamma  # New parameter
):
    """
    Finds the best portfolio for a given lambda by enumerating all K-stock combinations.
    """
    min_energy_full = float('inf')
    best_portfolio_indices = None

    # Iterate through all combinations of K stocks out of n total stocks.
    # Each combination represents the indices of stocks selected for the portfolio.
    for indices_tuple in itertools.combinations(range(n), K):
        # Create a binary portfolio vector 'x' from the selected indices.
        # All elements are 0 by default, then set 1 at selected indices.
        x = np.zeros(n, dtype=int)
        for idx in indices_tuple:
            x[idx] = 1

        # Evaluate the energy of the current portfolio
        current_energy = energy(
            x,
            mu,
            Sigma,
            current_lambda,
            alpha,
            K,
            sector_map,
            sector_cap,
            current_beta,
            prices,        # Pass new parameter
            budget,        # Pass new parameter
            current_gamma  # Pass new parameter
        )

        # If this portfolio has lower energy, it's our new best.
        if current_energy < min_energy_full:
            min_energy_full = current_energy
            best_portfolio_indices = indices_tuple

    # Convert best_portfolio_indices to a readable list of stock names (e.g., 'Stock_0', 'Stock_1')
    selected_stocks = [
        stock_names[i] for i in best_portfolio_indices
    ]
    return selected_stocks, min_energy_full

--- Full V1 Implementation (n=20 stocks, target K=6) ---


## Section 7: λ sweep

In [195]:
# Run full V2 implementation across lambda and beta values
# Results stored in pandas DataFrame

results = []

print("--- Starting Lambda + Beta + Gamma Sweep ---")

# beta_values = [0, 0.1, 1, 10] # Already defined globally

for current_lambda in lambda_values:
    for current_beta in beta_values:
        for current_gamma in gamma_values: # New loop for gamma

            print(
                f"\nOptimizing for "
                f"lambda = {current_lambda}, "
                f"beta = {current_beta}, "
                f"gamma = {current_gamma}..." # Updated print
            )

            # Find best portfolio
            selected_stocks, min_energy = find_best_portfolio_for_lambda(
                current_lambda=current_lambda,
                current_beta=current_beta,
                mu=mu,
                Sigma=Sigma,
                alpha=alpha,
                K=K,
                n=n,
                sector_map=sector_map,
                sector_cap=sector_cap, # Use variable
                stock_names=stocks,
                prices=prices,         # Pass new parameter
                budget=budget,         # Pass new parameter
                current_gamma=current_gamma # Pass new parameter
            )

            # Build binary selection vector x
            x = np.zeros(n)
            for idx, stock in enumerate(stocks):
                if stock in selected_stocks:
                    x[idx] = 1

            # Sector analysis
            sector_counts = get_sector_counts(x, sector_map)
            sector_violation = check_sector_violation(
                sector_counts,
                sector_cap=sector_cap # Use variable
            )

            # Budget analysis (New)
            total_portfolio_cost = np.dot(x, prices) if prices is not None else 0
            budget_violation_amount = max(0, total_portfolio_cost - budget) if budget is not None else 0
            budget_violation_status = "Violated" if budget_violation_amount > 0 else "None"


            # Store results
            results.append({
                'lambda': current_lambda,
                'beta': current_beta,
                'gamma': current_gamma, # New result field
                'selected_stocks': selected_stocks,
                'energy': min_energy,
                'sector_counts': sector_counts,
                'sector_violation': sector_violation,
                'total_portfolio_cost': total_portfolio_cost, # New result field
                'budget_violation': budget_violation_status, # New result field
                'budget_violation_amount': budget_violation_amount
            })

            # Clean readable output
            print(f"  Best portfolio: {selected_stocks}")
            print(f"  Energy: {min_energy:.6f}")
            print(f"  Sector counts: {sector_counts}")
            print(f"  Sector violation: {sector_violation}")
            print(f"  Total portfolio cost: ${total_portfolio_cost:.2f}") # New print
            print(f"  Budget violation: {budget_violation_status}") # New print

# Create results DataFrame
results_df = pd.DataFrame(results)

print("\n--- Lambda + Beta + Gamma Sweep Results ---") # Updated print
print(results_df.to_string(index=False))

--- Starting Lambda + Beta + Gamma Sweep ---

Optimizing for lambda = 0.1, beta = 0, gamma = 0...
  Best portfolio: ['AAPL', 'AMZN', 'MA', 'GS', 'ABBV', 'CVX']
  Energy: -0.015277
  Sector counts: {'Technology': np.float64(2.0), 'Finance': np.float64(2.0), 'Healthcare': np.float64(1.0), 'Industrial_Energy': np.float64(1.0)}
  Sector violation: None
  Total portfolio cost: $1910.00
  Budget violation: Violated

Optimizing for lambda = 0.1, beta = 0, gamma = 0.01...
  Best portfolio: ['AAPL', 'GOOGL', 'AMZN', 'MRK', 'ABBV', 'CVX']
  Energy: -0.012226
  Sector counts: {'Technology': np.float64(3.0), 'Finance': np.float64(0.0), 'Healthcare': np.float64(2.0), 'Industrial_Energy': np.float64(1.0)}
  Sector violation: Violated: {'Technology': np.float64(3.0)}
  Total portfolio cost: $1070.00
  Budget violation: None

Optimizing for lambda = 0.1, beta = 0, gamma = 0.1...
  Best portfolio: ['AAPL', 'GOOGL', 'AMZN', 'MRK', 'ABBV', 'CVX']
  Energy: -0.012226
  Sector counts: {'Technology': np.flo

In [196]:
print("\n--- V3 Key Observation ---")

violated_cases = results_df[
    results_df["budget_violation"] == "Violated"
]

feasible_cases = results_df[
    results_df["budget_violation"] == "None"
]

print(
    f"Cases violating budget: "
    f"{len(violated_cases)}"
)

print(
    f"Budget-feasible cases: "
    f"{len(feasible_cases)}"
)

print()

print(
    "Observation:"
)

print(
    "A small budget penalty "
    "(gamma = 0.01) was sufficient "
    "to enforce budget feasibility."
)

print()

print(
    "Increasing gamma further "
    "generally did not alter "
    "portfolio composition, "
    "suggesting stable convergence."
)


--- V3 Key Observation ---
Cases violating budget: 16
Budget-feasible cases: 48

Observation:
A small budget penalty (gamma = 0.01) was sufficient to enforce budget feasibility.

Increasing gamma further generally did not alter portfolio composition, suggesting stable convergence.


In [197]:
print("--- V3 Interpretation ---")

print(
    "When gamma = 0, "
    "the optimizer may exceed the budget "
    "if higher-cost portfolios improve "
    "risk-adjusted return."
)

print()

print(
    "When gamma > 0, "
    "the optimizer shifts toward "
    "budget-feasible portfolios."
)

print()

print(
    "In this experiment, "
    "the budget constraint activated "
    "on the real dataset and changed "
    "portfolio composition."
)

print()

print(
    "Interestingly, once a feasible "
    "budget-respecting portfolio was found, "
    "increasing gamma further often "
    "did not change the solution."
)

print()

print(
    "This suggests the selected "
    "budget level was meaningfully binding "
    "without over-constraining "
    "the optimization."
)

--- V3 Interpretation ---
When gamma = 0, the optimizer may exceed the budget if higher-cost portfolios improve risk-adjusted return.

When gamma > 0, the optimizer shifts toward budget-feasible portfolios.

In this experiment, the budget constraint activated on the real dataset and changed portfolio composition.

Interestingly, once a feasible budget-respecting portfolio was found, increasing gamma further often did not change the solution.

This suggests the selected budget level was meaningfully binding without over-constraining the optimization.


## Section 6 — Controlled Budget Stress Test (V3)

In [198]:
# ==========================================
# V3 TEST GROUND — FORCE BUDGET CONSTRAINT
# ==========================================

print("\n--- V3 Controlled Budget Stress Test ---")

# Select an intentionally small subset of stocks for clarity
stress_test_indices = [0, 1, 2, 5, 10, 15] # Original stocks: AAPL, MSFT, NVDA, JPM, LLY, XOM
stress_test_stocks = [stocks[i] for i in stress_test_indices]

# Create a reduced mu and Sigma for the test
mu_stress = mu.iloc[stress_test_indices].copy()
Sigma_stress = Sigma.iloc[stress_test_indices, stress_test_indices].copy()

# Artificially assign high prices to a few stocks to create a budget challenge
prices_stress = np.array([
    500.0,  # AAPL (index 0 in stress_test_stocks)
    600.0,  # MSFT (index 1)
    700.0,  # NVDA (index 2)
    150.0,  # JPM (index 3)
    120.0,  # LLY (index 4)
    100.0   # XOM (index 5)
])

# Artificially boost returns for the expensive tech stocks to ensure they are chosen initially
# This is to force a budget violation when gamma=0
mu_stress.iloc[0] = 0.050  # AAPL (index 0, price 500)
mu_stress.iloc[1] = 0.045  # MSFT (index 1, price 600)
mu_stress.iloc[2] = 0.040  # NVDA (index 2, price 700)
mu_stress.iloc[3] = 0.001  # JPM (index 3, price 150)
mu_stress.iloc[4] = 0.001  # LLY (index 4, price 120)
mu_stress.iloc[5] = 0.001  # XOM (index 5, price 100)

# Define test parameters
K_stress = 3 # Select 3 stocks
n_stress = len(stress_test_indices)
lambda_stress = 0.01 # Keep lambda very low to prioritize boosted returns
beta_stress = 0 # Disable sector penalty for this test
alpha_stress = 20 # Keep cardinality penalty
sector_map_stress = { # Simple map for the stress test stocks
    "Tech": [0, 1, 2],
    "Finance": [3],
    "Healthcare": [4],
    "Energy": [5]
}
sector_cap_stress = 1 # Allow only 1 stock per sector to keep it simple

# Define a tight budget that expensive stocks will violate (500+600+700 = 1800)
budget_stress = 1200.0 # This will be violated by the expensive tech stocks

print(f"Test stocks: {stress_test_stocks}")
print(f"Test prices: {prices_stress}")
print(f"K = {K_stress}")
print(f"Budget = ${budget_stress:.2f}")
print(f"Modified mu_stress (top 3 highly attractive):\n{mu_stress}")

# Compare gamma values: 0 (no penalty) vs. 1 (with penalty)
stress_gamma_values = [0, 1]

for gamma_stress in stress_gamma_values:
    print("\n" + "="*50)
    print(f"Running Stress Test for Gamma = {gamma_stress}")
    print("="*50)

    selected_stocks_stress, min_energy_stress = find_best_portfolio_for_lambda(
        current_lambda=lambda_stress,
        current_beta=beta_stress,
        mu=mu_stress,
        Sigma=Sigma_stress,
        alpha=alpha_stress,
        K=K_stress,
        n=n_stress,
        sector_map=sector_map_stress,
        sector_cap=sector_cap_stress,
        stock_names=stress_test_stocks,
        prices=prices_stress,
        budget=budget_stress,
        current_gamma=gamma_stress
    )

    # Build x vector for analysis
    x_stress = np.zeros(n_stress)
    for idx, stock in enumerate(stress_test_stocks):
        if stock in selected_stocks_stress:
            x_stress[idx] = 1

    total_portfolio_cost_stress = np.dot(x_stress, prices_stress)
    budget_violation_status_stress = "Violated" if total_portfolio_cost_stress > budget_stress else "None"

    print("Selected Portfolio:")
    print(selected_stocks_stress)

    print("\nEnergy:")
    print(round(min_energy_stress, 6))

    print("\nTotal Portfolio Cost:")
    print(f"${total_portfolio_cost_stress:.2f}")

    print("\nBudget Violation Status:")
    print(budget_violation_status_stress)

    if gamma_stress == 0 and budget_violation_status_stress == "Violated":
        print("Expected: With gamma=0, budget constraint is ignored, leading to violation.")
    elif gamma_stress > 0 and budget_violation_status_stress == "None":
        print("Expected: With gamma > 0, the budget constraint forces a compliant portfolio.")
    elif gamma_stress > 0 and budget_violation_status_stress == "Violated":
        print("Note: A violation can still occur if gamma is not high enough to fully penalize the overrun.")
    else:
        print("Note: Portfolio naturally respected budget or gamma effect was minimal.")


--- V3 Controlled Budget Stress Test ---
Test stocks: ['AAPL', 'MSFT', 'NVDA', 'JPM', 'LLY', 'XOM']
Test prices: [500. 600. 700. 150. 120. 100.]
K = 3
Budget = $1200.00
Modified mu_stress (top 3 highly attractive):
Ticker
AAPL    0.050
ABBV    0.045
AMZN    0.040
CVX     0.001
JNJ     0.001
MSFT    0.001
Name: expected_return, dtype: float64

Running Stress Test for Gamma = 0
Selected Portfolio:
['AAPL', 'MSFT', 'NVDA']

Energy:
-0.134992

Total Portfolio Cost:
$1800.00

Budget Violation Status:
Violated
Expected: With gamma=0, budget constraint is ignored, leading to violation.

Running Stress Test for Gamma = 1
Selected Portfolio:
['AAPL', 'MSFT', 'XOM']

Energy:
-0.095993

Total Portfolio Cost:
$1200.00

Budget Violation Status:
None
Expected: With gamma > 0, the budget constraint forces a compliant portfolio.


## Section 8: Classical Markowitz Baseline

In [210]:
from scipy.optimize import minimize

print("Imported scipy.optimize for Markowitz baseline.")

Imported scipy.optimize for Markowitz baseline.


In [211]:
def solve_markowitz(mu, Sigma, lam):
    """
    Solves the classical Markowitz portfolio optimization problem.

    Maximizes: mu^T w - lambda * w^T Sigma w
    Subject to: sum(w_i) = 1, w_i >= 0 (no short selling)

    Args:
        mu (pd.Series): Expected returns vector.
        Sigma (pd.DataFrame): Covariance matrix.
        lam (float): Risk aversion parameter.

    Returns:
        tuple: (optimal_weights, expected_return, variance)
    """
    num_assets = len(mu)

    # The objective function to minimize (negative of the Markowitz objective)
    def objective_function(w):
        portfolio_return = np.dot(mu, w)
        portfolio_variance = np.dot(w.T, np.dot(Sigma, w))
        return -(portfolio_return - lam * portfolio_variance)

    # Constraints
    # Sum of weights must be 1
    constraints = ({
        'type': 'eq',
        'fun': lambda w: np.sum(w) - 1
    })

    # Weights must be non-negative (no short selling)
    bounds = tuple((0, 1) for _ in range(num_assets)) # Weights between 0 and 1

    # Initial guess (equal weighting)
    initial_weights = np.array(num_assets * [1. / num_assets])

    # Solve the optimization problem
    result = minimize(
        objective_function,
        initial_weights,
        method='SLSQP',
        bounds=bounds,
        constraints=constraints
    )

    if result.success:
        optimal_weights = result.x
        expected_return = np.dot(mu, optimal_weights)
        variance = np.dot(optimal_weights.T, np.dot(Sigma, optimal_weights))
        return optimal_weights, expected_return, variance
    else:
        raise Exception("Markowitz optimization failed: " + result.message)

print("Markowitz solver function defined.")

Markowitz solver function defined.


In [212]:
markowitz_results = []

print("--- Running Classical Markowitz Baseline Sweep ---")

for lam in lambda_values:
    print(f"Solving Markowitz for lambda = {lam}...")
    try:
        weights, ret, var = solve_markowitz(mu, Sigma, lam)
        markowitz_results.append({
            'lambda': lam,
            'expected_return': ret,
            'variance': var,
            'weights': weights.round(4)
        })
    except Exception as e:
        print(f"Error for lambda {lam}: {e}")

markowitz_df = pd.DataFrame(markowitz_results)

print("\n--- Markowitz Results DataFrame ---")
print(markowitz_df.to_string(index=False))

--- Running Classical Markowitz Baseline Sweep ---
Solving Markowitz for lambda = 0.1...
Solving Markowitz for lambda = 0.5...
Solving Markowitz for lambda = 1.0...
Solving Markowitz for lambda = 2.0...

--- Markowitz Results DataFrame ---
 lambda  expected_return  variance                                                                                                    weights
    0.1         0.003790  0.000285 [0.0, 0.0, 0.0, 0.0, 0.6052, 0.0, 0.0, 0.3948, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
    0.5         0.003786  0.000283 [0.0, 0.0, 0.0, 0.0, 0.5993, 0.0, 0.0, 0.4007, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
    1.0         0.003782  0.000282 [0.0, 0.0, 0.0, 0.0, 0.5917, 0.0, 0.0, 0.4083, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
    2.0         0.003773  0.000279 [0.0, 0.0, 0.0, 0.0, 0.5759, 0.0, 0.0, 0.4241, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


In [213]:
print("\n--- Top 5 Assets by Weight for Each Lambda (Markowitz) ---")

top_weights_data = []

for _, row in markowitz_df.iterrows():
    current_lambda = row['lambda']
    weights = row['weights']

    # Create a Series of weights with stock names as index
    weights_series = pd.Series(weights, index=stocks)

    # Get top 5 assets by weight
    top_5_assets = weights_series.nlargest(5)

    for asset, weight in top_5_assets.items():
        if weight > 0: # Only include assets with non-zero weight
            top_weights_data.append({
                'lambda': current_lambda,
                'asset': asset,
                'weight': weight
            })

top_weights_df = pd.DataFrame(top_weights_data)
print(top_weights_df.to_string(index=False))


--- Top 5 Assets by Weight for Each Lambda (Markowitz) ---
 lambda asset  weight
    0.1  AMZN  0.6052
    0.1    MA  0.3948
    0.5  AMZN  0.5993
    0.5    MA  0.4007
    1.0  AMZN  0.5917
    1.0    MA  0.4083
    2.0  AMZN  0.5759
    2.0    MA  0.4241


## Section 9: Markowitz vs V3 Comparison

In [214]:
print("--- Markowitz vs V3 Comparison ---")

# SECTION 9.1: Select ONE representative V3 portfolio
print("\nSelecting representative V3 portfolio (lambda=1.0, beta=1, gamma=0.01)...")
v3_lambda = 1.0
v3_beta = 1
v3_gamma = 0.01

v3_portfolio_data = results_df[
    (results_df['lambda'] == v3_lambda) &
    (results_df['beta'] == v3_beta) &
    (results_df['gamma'] == v3_gamma)
].iloc[0] # Get the first (and should be only) matching row

selected_v3_stocks = v3_portfolio_data['selected_stocks']
print(f"V3 Portfolio: {selected_v3_stocks}")

# SECTION 9.2: Compute V3 metrics
v3_x = np.zeros(len(stocks))
for stock_name in selected_v3_stocks:
    idx = stocks.index(stock_name)
    v3_x[idx] = 1

v3_expected_return = np.dot(mu, v3_x)
v3_variance = np.dot(v3_x.T, np.dot(Sigma, v3_x))
v3_portfolio_cost = np.dot(v3_x, prices)
v3_number_of_assets = np.sum(v3_x)
v3_sector_violation = v3_portfolio_data['sector_violation']
v3_budget_violation = v3_portfolio_data['budget_violation']

print("\n--- V3 Portfolio Metrics ---")
print(f"Expected Return: {v3_expected_return:.6f}")
print(f"Variance: {v3_variance:.6f}")
print(f"Number of Assets: {int(v3_number_of_assets)}")
print(f"Portfolio Cost: ${v3_portfolio_cost:.2f}")
print(f"Sector Violation: {v3_sector_violation}")
print(f"Budget Violation: {v3_budget_violation}")

--- Markowitz vs V3 Comparison ---

Selecting representative V3 portfolio (lambda=1.0, beta=1, gamma=0.01)...
V3 Portfolio: ['AAPL', 'AMZN', 'JPM', 'BAC', 'ABBV', 'CVX']

--- V3 Portfolio Metrics ---
Expected Return: 0.011924
Variance: 0.002726
Number of Assets: 6
Portfolio Cost: $1075.00
Sector Violation: None
Budget Violation: None


In [215]:
# SECTION 9.3: Compute corresponding Markowitz metrics
print("\nExtracting Markowitz metrics for lambda=1.0...")
markowitz_comparison_data = markowitz_df[markowitz_df['lambda'] == v3_lambda].iloc[0]

markowitz_expected_return = markowitz_comparison_data['expected_return']
markowitz_variance = markowitz_comparison_data['variance']
markowitz_weights = markowitz_comparison_data['weights']

# For Markowitz, 'number of assets' usually refers to non-zero weights, but it's fractional
markowitz_number_of_assets = np.sum(markowitz_weights > 1e-6) # Count effectively non-zero weights

# For budget and sector constraints, Markowitz does not explicitly enforce them in this setup
# We'll calculate a 'cost' for comparison, but without an explicit constraint.
markowitz_portfolio_cost = np.dot(markowitz_weights, prices) # Calculate notional cost

print("\n--- Markowitz Portfolio Metrics (lambda=1.0) ---")
print(f"Expected Return: {markowitz_expected_return:.6f}")
print(f"Variance: {markowitz_variance:.6f}")
print(f"Number of Assets (effectively non-zero): {int(markowitz_number_of_assets)}")
print(f"Notional Portfolio Cost: ${markowitz_portfolio_cost:.2f}")


Extracting Markowitz metrics for lambda=1.0...

--- Markowitz Portfolio Metrics (lambda=1.0) ---
Expected Return: 0.003782
Variance: 0.000282
Number of Assets (effectively non-zero): 2
Notional Portfolio Cost: $342.49


In [216]:
# SECTION 9.4: Create comparison table
comparison_data = [
    {'Metric': 'Expected Return', 'Markowitz': f"{markowitz_expected_return:.6f}", 'V3': f"{v3_expected_return:.6f}"},
    {'Metric': 'Variance', 'Markowitz': f"{markowitz_variance:.6f}", 'V3': f"{v3_variance:.6f}"},
    {'Metric': 'Number of Assets', 'Markowitz': f"{int(markowitz_number_of_assets)}", 'V3': f"{int(v3_number_of_assets)}"},
    {'Metric': 'Portfolio Cost', 'Markowitz': f"${markowitz_portfolio_cost:.2f}", 'V3': f"${v3_portfolio_cost:.2f}"},
    {'Metric': 'Sector Constraint', 'Markowitz': 'N/A (not enforced)', 'V3': v3_sector_violation},
    {'Metric': 'Budget Constraint', 'Markowitz': f"N/A (not enforced, cost: ${markowitz_portfolio_cost:.2f})", 'V3': v3_budget_violation}
]

comparison_df = pd.DataFrame(comparison_data)

print("\n--- Markowitz vs V3 Comparison Table ---")
print(comparison_df.to_string(index=False))


--- Markowitz vs V3 Comparison Table ---
           Metric                         Markowitz       V3
  Expected Return                          0.003782 0.011924
         Variance                          0.000282 0.002726
 Number of Assets                                 2        6
   Portfolio Cost                           $342.49 $1075.00
Sector Constraint                N/A (not enforced)     None
Budget Constraint N/A (not enforced, cost: $342.49)     None


In [217]:
# SECTION 9.5: Add interpretation cell
print("\n--- Interpretation: Markowitz vs V3 ---")

print("1.  **Allocation Types:**")
print("    The Classical Markowitz model, as implemented here, uses continuous (fractional) asset allocations, allowing for precise weighting of each asset. In contrast, V3 employs a binary (0 or 1) asset selection, meaning assets are either fully included or excluded from the portfolio, without fractional ownership.")

print("2.  **Cardinality Constraint:**")
print("    Markowitz does not inherently enforce a specific number of assets (cardinality); it will allocate weights to as many assets as it deems optimal. V3, however, strictly enforces a cardinality constraint (K), selecting exactly K assets, which is a common real-world requirement.")

print("3.  **Additional Constraints in V3:**")
print("    V3 extends the Markowitz-like objective function with soft penalties for sector exposure and budget limits. These are not present in the basic Markowitz formulation and allow V3 to address more complex, practical portfolio construction rules.")

print("4.  **Performance Differences:**")
print("    Given that V3 solves a more constrained problem (binary selection, cardinality, sector, and budget), it is expected that its 'optimal' performance (e.g., expected return or variance) might differ from the unconstrained Markowitz solution. Markowitz provides a theoretical upper bound for return-risk efficiency under its assumptions, while V3 aims for a practical solution adhering to real-world limitations. Small performance differences are therefore a natural consequence of the added constraints.")


--- Interpretation: Markowitz vs V3 ---
1.  **Allocation Types:**
    The Classical Markowitz model, as implemented here, uses continuous (fractional) asset allocations, allowing for precise weighting of each asset. In contrast, V3 employs a binary (0 or 1) asset selection, meaning assets are either fully included or excluded from the portfolio, without fractional ownership.
2.  **Cardinality Constraint:**
    Markowitz does not inherently enforce a specific number of assets (cardinality); it will allocate weights to as many assets as it deems optimal. V3, however, strictly enforces a cardinality constraint (K), selecting exactly K assets, which is a common real-world requirement.
3.  **Additional Constraints in V3:**
    V3 extends the Markowitz-like objective function with soft penalties for sector exposure and budget limits. These are not present in the basic Markowitz formulation and allow V3 to address more complex, practical portfolio construction rules.
4.  **Performance Differ

## Section 8: Simple interpretation

In [199]:
print("--- Interpretation of Results ---")

# Access the first row (lowest lambda) and last row (highest lambda) for comparison
lowest_lambda_result = results_df.iloc[0]
highest_lambda_result = results_df.iloc[-1]

print(f"1. **Low Lambda (e.g., \u03BB = {lowest_lambda_result['lambda']}):**")
print(f"   At low \u03BB, the objective function places more weight on **maximizing expected return**.")
print(f"   The selected portfolio ({lowest_lambda_result['selected_stocks']}) will likely contain stocks with high individual expected returns, potentially accepting higher risk.")

print(f"\n2. **High Lambda (e.g., \u03BB = {highest_lambda_result['lambda']}):**")
print(f"   At high \u03BB, the objective function places more emphasis on **minimizing portfolio risk (covariance)**.")
print(f"   The selected portfolio ({highest_lambda_result['selected_stocks']}) will aim for lower overall variance, likely selecting stocks that are less correlated or individually less volatile.")

print("\nIn general, increasing \u03BB shifts the trade-off from return maximization towards risk minimization within the fixed cardinality constraint (K). This demonstrates the risk-aversion parameter's role in shaping the optimal portfolio composition.")

--- Interpretation of Results ---
1. **Low Lambda (e.g., λ = 0.1):**
   At low λ, the objective function places more weight on **maximizing expected return**.
   The selected portfolio (['AAPL', 'AMZN', 'MA', 'GS', 'ABBV', 'CVX']) will likely contain stocks with high individual expected returns, potentially accepting higher risk.

2. **High Lambda (e.g., λ = 2.0):**
   At high λ, the objective function places more emphasis on **minimizing portfolio risk (covariance)**.
   The selected portfolio (['AAPL', 'AMZN', 'JPM', 'BAC', 'ABBV', 'CVX']) will aim for lower overall variance, likely selecting stocks that are less correlated or individually less volatile.

In general, increasing λ shifts the trade-off from return maximization towards risk minimization within the fixed cardinality constraint (K). This demonstrates the risk-aversion parameter's role in shaping the optimal portfolio composition.


#Conformance Tests

In [200]:
print(mu.shape)
print(Sigma.shape)

(20,)
(20, 20)


In [201]:
print(mu.isna().sum())
print(Sigma.isna().sum().sum())

0
0


# TOY VERIFICATION (5 STOCKS)
#

In [202]:
small_n = 5
small_indices = [0, 5, 10, 15, 18]

In [203]:
mu_small = mu.iloc[small_indices]
Sigma_small = Sigma.iloc[
    small_indices,
    small_indices
]

stocks_small = [stocks[i] for i in small_indices]

Step 2: Freeze small problem

In [204]:
K_small = 2
lam_Small = 0

Step 3: Enumerate ALL possibilities

In [205]:
all_portfolios = list(
    itertools.product([0,1], repeat=small_n)
)

Step 4: Evaluate all portfolios

In [206]:
best_energy = float('inf')
best_portfolio = None

In [207]:
for portfolio in all_portfolios:

    portfolio = np.array(portfolio)

    e = energy(
        portfolio,
        mu_small,
        Sigma_small,
        lam_small,
        alpha,
        K_small
    )

    if e < best_energy:
        best_energy = e
        best_portfolio = portfolio

Step 5: Decode selected stocks

In [208]:
selected_indices = np.where(best_portfolio == 1)[0]

In [225]:
selected_stocks = [
    stocks[i] for i in selected_indices
]

Step 6: Print results

In [226]:
print("TOY VERIFICATION RESULTS")
print("="*40)

print("Selected stocks:")
print(selected_stocks)

print("\nCardinality:")
print(np.sum(best_portfolio))

print("\nEnergy:")
print(best_energy)

TOY VERIFICATION RESULTS
Selected stocks:
['AAPL', 'NVDA']

Cardinality:
2

Energy:
-0.0033474261970486362


In [227]:
print("Mean Returns (mu):")
for stock, val in zip(stocks_small, mu_small):
    print(f"{stock}: {val:.6f}")

print("\nDiagonal of Covariance Matrix (risk proxy):")
for stock, val in zip(stocks_small, np.diag(Sigma_small)):
    print(f"{stock}: {val:.6f}")

Mean Returns (mu):
AAPL: 0.001866
JPM: 0.001389
LLY: 0.001648
XOM: 0.000050
CAT: -0.000325

Diagonal of Covariance Matrix (risk proxy):
AAPL: 0.000194
JPM: 0.000194
LLY: 0.000111
XOM: 0.000240
CAT: 0.000194


# V2 Sector Constraint Test Ground

In [228]:
# ==========================================
# V2 TEST GROUND — FORCE SECTOR CONSTRAINT
# ==========================================

print("\n--- V2 Sector Constraint Test Ground ---")

# Select intentionally tech-heavy subset
test_indices = [0, 1, 2, 4, 5, 10]
# AAPL, MSFT, NVDA, AMZN, JPM, LLY

test_stocks = [stocks[i] for i in test_indices]

# Reduced mu and Sigma
mu_test = mu.iloc[test_indices].copy()

# Artificially boost tech returns
mu_test.iloc[0] = 0.010  # AAPL
mu_test.iloc[1] = 0.011  # MSFT
mu_test.iloc[2] = 0.012  # NVDA
mu_test.iloc[3] = 0.009  # AMZN

# Lower non-tech attractiveness
mu_test.iloc[4] = 0.001  # JPM
mu_test.iloc[5] = 0.001  # LLY

Sigma_test = Sigma.iloc[
    test_indices,
    test_indices
]

# Local sector map for toy experiment
sector_map_test = {
    "Technology": [0, 1, 2, 3],  # AAPL, MSFT, NVDA, AMZN
    "Finance": [4],              # JPM
    "Healthcare": [5]            # LLY
}

# Freeze parameters
K_test = 4
n_test = len(test_indices)
lambda_test = 0.1
sector_cap_test = 2

print(f"Test stocks: {test_stocks}")
print(f"K = {K_test}")
print(f"Sector cap = {sector_cap_test}")

# Compare beta values
test_beta_values = [0, 10]

for beta_test in test_beta_values:

    selected_stocks, min_energy = find_best_portfolio_for_lambda(
        current_lambda=lambda_test,
        current_beta=beta_test,
        mu=mu_test,
        Sigma=Sigma_test,
        alpha=alpha,
        K=K_test,
        n=n_test,
        sector_map=sector_map_test,
        sector_cap=sector_cap_test,
        stock_names=test_stocks
    )

    # Build x vector
    x_test = np.zeros(n_test)

    for idx, stock in enumerate(test_stocks):
        if stock in selected_stocks:
            x_test[idx] = 1

    # Sector analysis
    sector_counts = get_sector_counts(
        x_test,
        sector_map_test
    )

    violation_status = check_sector_violation(
        sector_counts,
        sector_cap_test
    )

    print("\n" + "="*50)
    print(f"Beta = {beta_test}")
    print("="*50)

    print("Selected Portfolio:")
    print(selected_stocks)

    print("\nEnergy:")
    print(round(min_energy, 6))

    print("\nSector Counts:")
    print(sector_counts)

    print("\nSector Violation:")
    print(violation_status)


--- V2 Sector Constraint Test Ground ---
Test stocks: ['AAPL', 'MSFT', 'NVDA', 'AMZN', 'JPM', 'LLY']
K = 4
Sector cap = 2


TypeError: find_best_portfolio_for_lambda() missing 3 required positional arguments: 'prices', 'budget', and 'current_gamma'

## Section 1 — Define Budget Parameters (V3)

In [222]:
# Define a frozen stock price vector
# Using placeholder values for the 20 stocks. In a real scenario, these would come from market data.
prices = np.array([
    150.0, 300.0, 600.0, 120.0, 180.0,  # Technology (AAPL, MSFT, NVDA, GOOGL, AMZN)
    140.0, 200.0, 350.0, 400.0, 100.0,  # Finance (JPM, V, MA, GS, BAC)
    110.0, 170.0, 80.0,  90.0, 130.0,   # Healthcare (LLY, JNJ, MRK, PFE, ABBV)
    70.0,  95.0,  160.0, 250.0, 220.0   # Industrial_Energy (XOM, CVX, GE, CAT, HON)
])

# Define the total budget. Intentionally set so it can sometimes bind for K=6.
# Average price is around $180-$200, so 6 stocks * 180 = $1080. A budget of 1000-1100 will bind.
budget = 1050.0

# Define gamma_values for the sweep, as the budget penalty coefficient
# gamma_values = [0, 0.01, 0.1, 1]
gamma_values = [0, 0.01, 0.1, 1]

print(f"Stock prices defined (sample): {prices[:5]}...")
print(f"Total budget (B): ${budget:.2f}")
print(f"Gamma values for sweep: {gamma_values}")

Stock prices defined (sample): [150. 300. 600. 120. 180.]...
Total budget (B): $1050.00
Gamma values for sweep: [0, 0.01, 0.1, 1]


## Section 2 — Budget Penalty Function (V3)

In [223]:
def budget_penalty(x, prices, budget, gamma):
    """
    Computes the budget constraint penalty for a given portfolio x.

    P_budget(x) = gamma * max(0, sum(w_i * x_i) - B)^2

    Args:
        x (np.array): A binary vector representing the portfolio.
        prices (np.array): A vector of stock prices corresponding to x.
        budget (float): The maximum allowed total cost of the portfolio.
        gamma (float): The budget penalty coefficient.

    Returns:
        float: The computed budget penalty.
    """
    # Calculate the total cost of the selected portfolio
    total_cost = np.dot(x, prices)

    # Calculate the violation: how much the total cost exceeds the budget
    violation = max(0, total_cost - budget)

    # Apply the soft quadratic penalty
    penalty = gamma * (violation**2)

    return penalty

## Section 3 — Real Dataset Interpretation Cell (V3)

In [224]:
print("--- Interpretation of Budget Constraint Sweep ---")
print("1. **Impact of Gamma:**")
print("   The `gamma` parameter controls the strength of the budget penalty. When `gamma = 0`, the budget constraint is effectively ignored, and the optimizer will select stocks based purely on return, risk, and sector constraints, potentially leading to portfolios that exceed the budget.")
print("   As `gamma` increases, exceeding the budget becomes more costly in terms of energy. This forces the optimization to favor portfolios that either stay within the budget or violate it by a smaller margin, even if it means selecting stocks with slightly lower returns or higher risk.")

print("\n2. **Constraint Not Always Binding:**")
print("   It's important to note that even with `gamma > 0`, you might observe that some portfolios do not violate the budget constraint. This does not imply a failure of the constraint or the optimization.")
print("   Instead, it often means that the optimal portfolio (considering returns, risk, cardinality, and sector constraints) naturally falls within the specified budget. In such cases, increasing `gamma` further might not change the portfolio composition, as there's no budget violation to penalize.")

print("\n3. **Trade-offs:**")
print("   Introducing a budget constraint adds another layer of complexity and trade-offs. A strict budget (high `gamma` or low `budget` value) might prevent the selection of otherwise highly attractive stocks, forcing the optimizer to find a 'good enough' portfolio that respects the financial limitations.")

--- Interpretation of Budget Constraint Sweep ---
1. **Impact of Gamma:**
   The `gamma` parameter controls the strength of the budget penalty. When `gamma = 0`, the budget constraint is effectively ignored, and the optimizer will select stocks based purely on return, risk, and sector constraints, potentially leading to portfolios that exceed the budget.
   As `gamma` increases, exceeding the budget becomes more costly in terms of energy. This forces the optimization to favor portfolios that either stay within the budget or violate it by a smaller margin, even if it means selecting stocks with slightly lower returns or higher risk.

2. **Constraint Not Always Binding:**
   It's important to note that even with `gamma > 0`, you might observe that some portfolios do not violate the budget constraint. This does not imply a failure of the constraint or the optimization.
   Instead, it often means that the optimal portfolio (considering returns, risk, cardinality, and sector constraints) na